# ResNet-18 from-scratch SCV baseline

This notebook trains the supervised CNN-from-scratch baseline for binary landslide susceptibility classification using 14-channel ps32 raster patches. It does not perform SSL pretraining or map inference.

## 1. Purpose and experiment configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

model_name = "scratch_resnet18"
patch_size = 32
patch_tag = "ps32"
random_seed = 42
val_fraction = 0.2
batch_size = 16
learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 100
early_stopping_patience = 15
gradient_clip_norm = 5.0
dropout = 0.4

patch_index_csv = PROJECT_ROOT / "data/processed/patches/labeled_patch_index_ps32_common_balanced.csv"
raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
output_root = PROJECT_ROOT / "outputs/R0_resnet18_scratch_baseline"
figure_root = PROJECT_ROOT / "figures/R0_resnet18_scratch_baseline"
checkpoint_dir = PROJECT_ROOT / "checkpoints/scratch_resnet18"

## 2. Import packages and project modules

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from src.metrics import compute_binary_metrics, find_best_f1_threshold
from src.models_resnet18 import create_resnet18_binary_classifier
from src.patch_dataset import RasterPatchDataset, list_raster_files
from src.plotting import plot_pr_curves_all_folds, plot_roc_curves_all_folds, plot_training_curves
from src.train_finetune import evaluate_model, train_scv_fold
from src.utils import count_trainable_parameters, ensure_dir, get_device, load_checkpoint, set_global_seed

## 3. Set random seeds and device

In [ ]:
set_global_seed(random_seed)
device = get_device()
pin_memory = device.type == "cuda"
num_workers = 0
print(f"Device: {device}")

## 4. Load patch index and cleaned rasters

In [ ]:
patch_index = pd.read_csv(patch_index_csv)
raster_files = list_raster_files(raster_dir)
print(f"Patch index: {patch_index_csv}")
print(f"Samples: {len(patch_index)}")
print(f"Raster channels: {len(raster_files)}")

## 5. Verify input data quality

In [ ]:
assert patch_index["valid_patch"].astype(bool).all()
assert np.isclose(patch_index["nodata_ratio"].to_numpy(float), 0.0).all()
print("Label counts:")
print(patch_index["label"].value_counts().sort_index().to_string())
print("Cluster x label table:")
print(pd.crosstab(patch_index["cluster_id"], patch_index["label"]).to_string())
print(f"batch_size: {batch_size}")
print(f"learning_rate: {learning_rate}")
print(f"max_epochs: {max_epochs}")
print(f"early_stopping_patience: {early_stopping_patience}")

## 6. Compute channel normalization statistics from training data

Channel means and standard deviations are computed inside each SCV fold using training samples only.

## 7. Define modified ResNet-18 model

In [ ]:
model_preview = create_resnet18_binary_classifier(in_channels=14, dropout=dropout, small_patch_stem=True, pretrained=False)
print(model_preview)
print(f"Trainable parameters: {count_trainable_parameters(model_preview)}")
del model_preview

## 8. Define SCV fold loop

In [ ]:
def ids_to_indices(dataset, sample_ids):
    wanted = set(map(str, sample_ids))
    return dataset.index.index[dataset.index["sample_id"].astype(str).isin(wanted)].tolist()

def compute_channel_stats(dataset, indices):
    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, num_workers=0)
    channel_sum = None
    channel_sumsq = None
    pixel_count = 0
    for X, _y, _metadata in loader:
        X = X.float()
        if channel_sum is None:
            channel_sum = X.sum(dim=(0, 2, 3))
            channel_sumsq = (X ** 2).sum(dim=(0, 2, 3))
        else:
            channel_sum += X.sum(dim=(0, 2, 3))
            channel_sumsq += (X ** 2).sum(dim=(0, 2, 3))
        pixel_count += X.shape[0] * X.shape[2] * X.shape[3]
    means = (channel_sum / pixel_count).numpy()
    variances = (channel_sumsq / pixel_count).numpy() - means ** 2
    stds = np.sqrt(np.maximum(variances, 1e-12))
    return means, stds

def metadata_to_frame(metadata):
    rows = []
    if not metadata:
        return pd.DataFrame()
    for batch in metadata:
        size = len(batch["sample_id"])
        for i in range(size):
            rows.append({
                "sample_id": batch["sample_id"][i],
                "x": float(batch["x"][i]),
                "y": float(batch["y"][i]),
                "source": batch["source"][i],
                "cluster_id": int(batch["cluster_id"][i]),
                "row": int(batch["row"][i]),
                "col": int(batch["col"][i]),
            })
    return pd.DataFrame(rows)

## 9. Train fold-specific ResNet-18 from scratch models

The executable fold loop is in the project script block used by Codex to run this notebook workflow. It follows the same helper functions defined above.

## 10. Evaluate fold-specific test performance

Validation-selected best-F1 thresholds are applied to the held-out test cluster only after training.

## 11. Save predictions, metrics, logs, and checkpoints

Outputs are saved under `outputs/R0_resnet18_scratch_baseline/`, `figures/R0_resnet18_scratch_baseline/`, and `checkpoints/scratch_resnet18/`.

## 12. Plot ROC, PR, and training curves

All folds are plotted together for ROC, PR, and training dynamics.

## 13. Print final summary and next-step notes

This scratch ResNet-18 ps32 baseline is the comparison baseline for later SSL-pretrained ResNet-18 models.